In [ ]:
# Databricks notebook source
# COMMAND ----------
# MAGIC %md
# MAGIC # Silver Table Declarative Pipeline: Socrata & Spatial Transformations
# MAGIC - Detects spatial data type dynamically (Socrata JSON, GeoJSON, WKT, Lat/Lon)
# MAGIC - Parses nested Socrata human_address components (street_address, city, state, zip)
# MAGIC - Drops internal `@computed_region*`, Socrata `:`, and raw location columns
# MAGIC - Emits iMAP-standard spatial columns: `geom` (4326), `geom_md` (6487), and `h3_r8`
# MAGIC - Preserves Bronze metadata, comments, and tags safely

# COMMAND ----------
import re
import dlt
from pyspark.sql import DataFrame
from pyspark.sql import functions as F

# -------------------------------------------------------------------------
# PIPELINE CONFIGURATION
# Add tables to process. Detection will resolve column names and format automatically.
# You can optionally set "preferred_loc_col" or "source_srid" if known.
# -------------------------------------------------------------------------
TABLE_CONFIGS = [
    {
        "bronze_table": "your_bronze_catalog.your_bronze_schema.your_real_bronze_table",
        "silver_table_name": "silver_sample_socrata",
        # Optional overrides:
        # "preferred_loc_col": "location",
        # "source_srid": 4326
    }
]

# -------------------------------------------------------------------------
# COMPONENT: AUTOMATIC LOCATION DETECTOR
# -------------------------------------------------------------------------
def detect_location_spec(bronze_df: DataFrame, preferred_loc_col: str = None) -> dict:
    """
    Inspects DataFrame schema and samples data to detect location type:
    - socrata_json (nested human_address / coordinates)
    - geojson (type + coordinates JSON)
    - wkt (Well-Known Text geometry string)
    - lat_lon (separate numeric coordinate columns)
    """
    cols = bronze_df.columns
    lower_col_map = {c.lower(): c for c in cols}

    # 1. Check for separate latitude/longitude columns
    lat_candidates = ["latitude", "lat", "y_coord", "y"]
    lon_candidates = ["longitude", "lon", "long", "x_coord", "x"]
    
    matched_lat = next((lower_col_map[c] for c in lat_candidates if c in lower_col_map), None)
    matched_lon = next((lower_col_map[c] for c in lon_candidates if c in lower_col_map), None)

    # 2. Check candidate location string columns
    candidate_cols = []
    if preferred_loc_col and preferred_loc_col in cols:
        candidate_cols.append(preferred_loc_col)
    
    common_names = ["location", "the_geom", "geom", "geometry", "wkt", "shape", "point"]
    for name in common_names:
        if name in lower_col_map and lower_col_map[name] not in candidate_cols:
            candidate_cols.append(lower_col_map[name])
            
    if not candidate_cols and not (matched_lat and matched_lon):
        candidate_cols = [f.name for f in bronze_df.schema.fields if f.dataType.simpleString() == "string"]

    # 3. Sample values to inspect format
    wkt_regex = re.compile(
        r"^(POINT|POLYGON|LINESTRING|MULTIPOINT|MULTIPOLYGON|MULTILINESTRING|GEOMETRYCOLLECTION)\s*\(", 
        re.IGNORECASE
    )

    for col_name in candidate_cols:
        sample_rows = (
            bronze_df.select(col_name)
            .filter(F.col(col_name).isNotNull() & (F.trim(F.col(col_name)) != ""))
            .limit(5)
            .collect()
        )
        if not sample_rows:
            continue

        sample_val = str(sample_rows[0][col_name]).strip()

        # Check Socrata JSON
        if ("human_address" in sample_val) or ("latitude" in sample_val and "longitude" in sample_val):
            return {
                "loc_type": "socrata_json",
                "raw_loc_col": col_name,
                "source_srid": 4326
            }

        # Check GeoJSON
        if '"type"' in sample_val and '"coordinates"' in sample_val:
            return {
                "loc_type": "geojson",
                "raw_loc_col": col_name,
                "source_srid": 4326
            }

        # Check WKT
        if wkt_regex.match(sample_val):
            return {
                "loc_type": "wkt",
                "raw_loc_col": col_name,
                "source_srid": 4326
            }

    # 4. Fallback to separate lat/lon if present
    if matched_lat and matched_lon:
        return {
            "loc_type": "lat_lon",
            "lon_col": matched_lon,
            "lat_col": matched_lat,
            "source_srid": 4326
        }

    raise ValueError(f"Unable to automatically detect location structure from columns: {cols}")

# -------------------------------------------------------------------------
# COMPONENT: SAFE METADATA EXTRACTION
# -------------------------------------------------------------------------
def get_bronze_metadata(bronze_table: str):
    """
    Extracts description and custom properties from the upstream bronze table.
    Filters system/Delta properties and renames .url keys to avoid UI auto-redaction.
    """
    # 1. Comment / Description
    tbl_desc = spark.sql(f"DESCRIBE TABLE EXTENDED {bronze_table}")
    comment_row = tbl_desc.filter(F.col("col_name") == "Comment").collect()
    table_comment = (
        comment_row[0]["data_type"] 
        if comment_row and comment_row[0]["data_type"] 
        else "Silver spatial table"
    )

    # 2. Table properties
    tbl_props_df = spark.sql(f"SHOW TBLPROPERTIES {bronze_table}")
    custom_props = {}
    system_prefixes = ("io.unitycatalog.", "delta.", "spark.", "pipelines.")
    
    for row in tbl_props_df.collect():
        key, val = row["key"], row["value"]
        if not key.startswith(system_prefixes):
            # Avoid Databricks auto-redacting keys containing '.url'
            if key.lower().endswith(".url"):
                custom_props[key.replace(".url", ".portal_link")] = val
            else:
                custom_props[key] = val

    return {
        "comment": table_comment,
        "properties": custom_props
    }

# -------------------------------------------------------------------------
# COMPONENT: TRANSFORMATION LOGIC
# -------------------------------------------------------------------------
def transform_to_silver(cfg: dict) -> DataFrame:
    bronze_table = cfg["bronze_table"]
    loc_type = cfg["loc_type"]
    srid = cfg.get("source_srid", 4326)
    raw_col = cfg.get("raw_loc_col")

    df = spark.table(bronze_table)

    # 1. Unpack location & construct base geometry __g
    if loc_type == "socrata_json":
        # Extract address components (without keeping standalone lat/long)
        df = (
            df.withColumn("street_address", F.get_json_object(F.get_json_object(F.col(raw_col), "$.human_address"), "$.address"))
              .withColumn("city", F.get_json_object(F.get_json_object(F.col(raw_col), "$.human_address"), "$.city"))
              .withColumn("state", F.get_json_object(F.get_json_object(F.col(raw_col), "$.human_address"), "$.state"))
              .withColumn("zip", F.get_json_object(F.get_json_object(F.col(raw_col), "$.human_address"), "$.zip"))
              .withColumn("__lat", F.expr(f"try_cast(get_json_object({raw_col}, '$.latitude') AS DOUBLE)"))
              .withColumn("__lon", F.expr(f"try_cast(get_json_object({raw_col}, '$.longitude') AS DOUBLE)"))
              .withColumn(
                  "__g",
                  F.expr(f"""
                      CASE 
                          WHEN __lon BETWEEN -180 AND 180 
                           AND __lat BETWEEN -90 AND 90 
                           AND NOT (__lon = 0 AND __lat = 0)
                          THEN st_point(__lon, __lat, {srid})
                          ELSE NULL 
                      END
                  """)
              )
        )
    elif loc_type == "geojson":
        df = df.withColumn(
            "__g",
            F.expr(f"""
                CASE 
                    WHEN {raw_col} IS NOT NULL AND length(trim({raw_col})) > 0
                    THEN st_setsrid(st_geomfromgeojson({raw_col}), {srid})
                    ELSE NULL 
                END
            """)
        )
    elif loc_type == "lat_lon":
        lon_col = cfg["lon_col"]
        lat_col = cfg["lat_col"]
        df = (
            df.withColumn("__lon", F.expr(f"try_cast({lon_col} AS DOUBLE)"))
              .withColumn("__lat", F.expr(f"try_cast({lat_col} AS DOUBLE)"))
              .withColumn(
                  "__g",
                  F.expr(f"""
                      CASE 
                          WHEN __lon BETWEEN -180 AND 180 
                           AND __lat BETWEEN -90 AND 90 
                           AND NOT (__lon = 0 AND __lat = 0)
                          THEN st_point(__lon, __lat, {srid})
                          ELSE NULL 
                      END
                  """)
              )
        )
    elif loc_type == "wkt":
        df = df.withColumn("__g", F.expr(f"st_setsrid(try_to_geometry({raw_col}), {srid})"))

    # 2. Derive iMAP standard spatial columns
    geom_expr = "__g" if srid == 4326 else "st_transform(__g, 4326)"
    geom_md_expr = "__g" if srid == 6487 else "st_transform(__g, 6487)"

    df = (
        df.withColumn("geom", F.expr(geom_expr))
          .withColumn("geom_md", F.expr(geom_md_expr))
          .withColumn(
              "h3_r8",
              F.expr("""
                  CASE 
                      WHEN geom IS NULL OR st_isempty(geom) THEN NULL
                      WHEN abs(st_x(st_centroid(geom))) > 180 OR abs(st_y(st_centroid(geom))) > 90 THEN NULL
                      ELSE h3_longlatash3(st_x(st_centroid(geom)), st_y(st_centroid(geom)), 8)
                  END
              """)
          )
    )

    # 3. Clean up columns:
    # - Internal scratch fields (__g, __lat, __lon)
    # - Raw location column
    # - Socrata @computed_region* columns
    # - Socrata internal system columns (starting with ':')
    drop_candidates = ["__g", "__lat", "__lon"]
    if raw_col:
        drop_candidates.append(raw_col)

    cols_to_drop = [
        c for c in df.columns
        if c in drop_candidates
        or c.startswith("@computed_region")
        or c.startswith(":")
    ]

    return df.drop(*cols_to_drop)

# -------------------------------------------------------------------------
# DLT FACTORY & RUNNER
# -------------------------------------------------------------------------
def create_silver_dlt_table(cfg: dict):
    table_name = cfg["silver_table_name"]
    bronze_table = cfg["bronze_table"]

    # 1. Fetch metadata & detect location schema from Bronze
    meta = get_bronze_metadata(bronze_table)
    bronze_df = spark.table(bronze_table)
    loc_spec = detect_location_spec(bronze_df, preferred_loc_col=cfg.get("preferred_loc_col"))
    
    # Merge detected specs (allowing manual config overrides)
    full_cfg = {**loc_spec, **cfg}

    # 2. Combine table properties with iMAP optimization settings
    merged_properties = {
        **meta["properties"],
        "delta.autoOptimize.optimizeWrite": "true",
        "delta.autoOptimize.autoCompact": "true",
        "delta.dataSkippingStatsColumns": "h3_r8"
    }

    # 3. Register table with data quality expectations
    @dlt.table(
        name=table_name,
        comment=meta["comment"],
        table_properties=merged_properties
    )
    @dlt.expect("centroid_h3_integrity", "geom IS NULL OR st_isempty(geom) OR h3_r8 IS NOT NULL")
    def silver_table():
        return transform_to_silver(full_cfg)

# Initialize all configured pipelines
for config in TABLE_CONFIGS:
    create_silver_dlt_table(config)